In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import sys

image_path = r"C:\INSI\Algèbre_linéaire\projetExamen\test.jpg"
mean = 0.0
std = 0.05

print("Demarrage du traitement...")
sys.stdout.flush()

def psnr(original, reconstructed):
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0
    return 20 * np.log10(max_pixel / np.sqrt(mse))

def mse_metric(original, reconstructed):
    return np.mean((original - reconstructed) ** 2)

def find_optimal_k(image, noisy_image, max_k=None, step=3):
    if max_k is None:
        max_k = min(noisy_image.shape[:2])
    
    k_values = range(5, min(max_k, 200), step)
    psnr_values = []
    mse_values = []
    
    total_k = len(list(k_values))
    
    for idx, k_test in enumerate(k_values):
        if idx % 10 == 0:
            progress = (idx / total_k) * 100
            print(f"  Progression: {progress:.0f}% ({idx}/{total_k})", end='\r')
            sys.stdout.flush()
        
        restored_test = restore_image_svd(noisy_image, k_test)
        psnr_val = psnr(image, restored_test)
        mse_val = mse_metric(image, restored_test)
        psnr_values.append(psnr_val)
        mse_values.append(mse_val)
    
    print(f"  Progression: 100% ({total_k}/{total_k})")
    sys.stdout.flush()
    
    optimal_idx_psnr = np.argmax(psnr_values)
    optimal_idx_mse = np.argmin(mse_values)
    optimal_k_psnr = list(k_values)[optimal_idx_psnr]
    optimal_k_mse = list(k_values)[optimal_idx_mse]
    
    return optimal_k_psnr, optimal_k_mse, k_values, psnr_values, mse_values

def restore_image_svd(noisy_image, k):
    restored = np.zeros_like(noisy_image)
    for c in range(3):
        U, S, Vt = np.linalg.svd(noisy_image[..., c], full_matrices=False)
        S_trunc = np.zeros_like(S)
        S_trunc[:k] = S[:k]
        restored[..., c] = U @ np.diag(S_trunc) @ Vt
    return np.clip(restored, 0.0, 1.0)

image = mpimg.imread(image_path)
if image.dtype != np.float32 and image.dtype != np.float64:
    image = image.astype(np.float32) / 255.0

print("Chargement de l'image... OK")
sys.stdout.flush()

noise = np.random.normal(mean, std, image.shape)
noisy_image = np.clip(image + noise, 0.0, 1.0)

print("Ajout du bruit... OK")
print("Recherche du k optimal en cours...")
sys.stdout.flush()

optimal_k_psnr, optimal_k_mse, k_values, psnr_values, mse_values = find_optimal_k(image, noisy_image, step=3)
restored_optimal = restore_image_svd(noisy_image, optimal_k_psnr)

print(f"k optimal trouve: {optimal_k_psnr}")
print("Calcul des metriques en cours...")
sys.stdout.flush()

psnr_noisy = psnr(image, noisy_image)
psnr_restored = psnr(image, restored_optimal)
mse_noisy = mse_metric(image, noisy_image)
mse_restored = mse_metric(image, restored_optimal)

k_compare = [10, optimal_k_psnr, 100]
results = {}

for k_val in k_compare:
    if k_val <= min(noisy_image.shape[:2]):
        restored = restore_image_svd(noisy_image, k_val)
        results[k_val] = {
            'psnr': psnr(image, restored),
            'mse': mse_metric(image, restored),
            'image': restored
        }

print("Calcul des metriques... OK")
print("\n" + "="*50)

print("\nImage bruitee:")
print(f"MSE  = {mse_noisy:.6f}")

print(f"\nImage restauree (k={optimal_k_psnr}):")
print(f"PSNR = {psnr_restored:.2f} dB")
print(f"MSE  = {mse_restored:.6f}")
print(f"Amelioration PSNR: +{psnr_restored - psnr_noisy:.2f} dB")
print(f"Reduction MSE: -{100 * (1 - mse_restored/mse_noisy):.1f}%")

print("\nComparaison pour differents k:")
print(f"{'k':^8} | {'PSNR (dB)':^12} | {'MSE':^12}")
print("-" * 40)
for k in k_compare:
    if k in results:
        psnr_val = results[k]['psnr']
        mse_val = results[k]['mse']
        print(f"{k:^8} | {psnr_val:^12.2f} | {mse_val:^12.6f}")

print(f"\nOptimal PSNR: {psnr_restored:.2f} dB (k={optimal_k_psnr})")
print(f"Optimal MSE: {mse_restored:.6f} (k={optimal_k_mse})")
print("="*50)

print("\nGeneration de la visualisation en cours...")
sys.stdout.flush()

fig = plt.figure(figsize=(18, 10))

plt.subplot(2, 3, 1)
plt.imshow(image)
plt.title("Image originale", fontsize=14, fontweight='bold')
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(noisy_image)
plt.title(f"Image bruitee\nPSNR = {psnr_noisy:.2f} dB\nMSE = {mse_noisy:.6f}", 
          fontsize=14, fontweight='bold')
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(restored_optimal)
plt.title(f"Image restauree (k={optimal_k_psnr})\nPSNR = {psnr_restored:.2f} dB\nMSE = {mse_restored:.6f}", 
          fontsize=14, fontweight='bold', color='green')
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_color('green')
    spine.set_linewidth(3)
plt.axis("off")

plt.subplot(2, 3, 4)
plt.plot(list(k_values), psnr_values, 'b-', linewidth=3, label='PSNR')
plt.axvline(x=optimal_k_psnr, color='green', linestyle='--', linewidth=3, 
            label=f'k optimal = {optimal_k_psnr}')
plt.scatter([optimal_k_psnr], [max(psnr_values)], color='green', s=400, zorder=5, 
            marker='*', edgecolors='black', linewidth=2)
plt.annotate(f'Optimal\nPSNR={max(psnr_values):.2f} dB', 
             xy=(optimal_k_psnr, max(psnr_values)), 
             xytext=(optimal_k_psnr + 25, max(psnr_values) - 1.5),
             arrowprops=dict(arrowstyle='->', color='green', lw=2.5),
             fontsize=11, fontweight='bold', color='green',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.9))
plt.xlabel('k (nombre de valeurs singulieres)', fontsize=12, fontweight='bold')
plt.ylabel('PSNR (dB)', fontsize=12, fontweight='bold')
plt.title('PSNR en fonction de k', fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='lower right')
plt.grid(True, alpha=0.3)

plt.subplot(2, 3, 5)
plt.plot(list(k_values), mse_values, 'r-', linewidth=3, label='MSE')
plt.axvline(x=optimal_k_mse, color='red', linestyle='--', linewidth=3, 
            label=f'k optimal MSE = {optimal_k_mse}')
plt.scatter([optimal_k_mse], [min(mse_values)], color='red', s=400, zorder=5, 
            marker='*', edgecolors='black', linewidth=2)
plt.annotate(f'Optimal MSE\nMSE={min(mse_values):.6f}', 
             xy=(optimal_k_mse, min(mse_values)), 
             xytext=(optimal_k_mse + 25, min(mse_values) + 0.0005),
             arrowprops=dict(arrowstyle='->', color='red', lw=2.5),
             fontsize=11, fontweight='bold', color='red',
             bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.9))
plt.xlabel('k (nombre de valeurs singulieres)', fontsize=12, fontweight='bold')
plt.ylabel('MSE', fontsize=12, fontweight='bold')
plt.title('MSE en fonction de k', fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='upper right')
plt.grid(True, alpha=0.3)

plt.subplot(2, 3, 6)
plt.axis('off')

table_data = [
    ["k", "PSNR (dB)", "MSE"],
    ["10", f"{results[10]['psnr']:.2f}", f"{results[10]['mse']:.6f}"],
    [str(optimal_k_psnr), f"{psnr_restored:.2f}", f"{mse_restored:.6f}"],
    ["100", f"{results[100]['psnr']:.2f}", f"{results[100]['mse']:.6f}"]
]

table = plt.table(cellText=table_data,
                  colLabels=None,
                  cellLoc='center',
                  loc='center',
                  bbox=[0.1, 0.2, 0.8, 0.6])

table.auto_set_font_size(False)
table.set_fontsize(14)

table[(0, 0)].set_facecolor('#2E86AB')
table[(0, 1)].set_facecolor('#2E86AB')
table[(0, 2)].set_facecolor('#2E86AB')
table[(0, 0)].set_text_props(color='white', fontweight='bold')
table[(0, 1)].set_text_props(color='white', fontweight='bold')
table[(0, 2)].set_text_props(color='white', fontweight='bold')

table[(2, 0)].set_facecolor('#C9E4C5')
table[(2, 1)].set_facecolor('#C9E4C5')
table[(2, 2)].set_facecolor('#C9E4C5')
table[(2, 0)].set_text_props(fontweight='bold')
table[(2, 1)].set_text_props(fontweight='bold')
table[(2, 2)].set_text_props(fontweight='bold')

for key, cell in table.get_celld().items():
    cell.set_edgecolor('black')
    cell.set_linewidth(1.5)
    if key[0] > 0:
        cell.set_text_props(fontsize=13)

plt.title('Comparaison pour differents k', fontsize=14, fontweight='bold', pad=20)

plt.suptitle(f'Restoration d\'Image par SVD - Resultats', 
             fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()

print("Visualisation prete!")
print("Affichage en cours...")
sys.stdout.flush()

plt.show()